# Defect-Type AUROC Overview

Class-level I-AUROC can hide which defect types are actually weak. This notebook recomputes defect-type AUROC across all available sample-score CSVs.

For each `model / dataset / class / defect_type`, AUROC is computed by comparing:

- all `good` samples from the same class
- samples from that single defect type

This is the overview to use when the question is robustness to specific failure modes, not class-level anomaly separability.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import roc_auc_score

ROOT = Path('..').resolve() if Path.cwd().name == 'notebooks' else Path('.').resolve()
RESULTS_ROOT = ROOT / 'results'
OUT_DIR = RESULTS_ROOT / 'defect_type_overview'
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_DIRS = {
    'adaptclip': RESULTS_ROOT / 'adaptclip',
    'anomalyclip': RESULTS_ROOT / 'anomalyclip',
    'winclip': RESULTS_ROOT / 'winclip',
}

REQUIRED_COLUMNS = {'dataset', 'class', 'label', 'image_score'}
MODEL_ORDER = ['adaptclip', 'anomalyclip', 'winclip']
DATASET_ORDER = ['mvtec', 'visa', 'btad', 'mpdd']
LOW_DEFECT_AUROC_THRESHOLD = 80.0

pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 220)


## 1. Collect Sample Score CSVs

In [ ]:
def has_required_columns(path):
    try:
        cols = set(pd.read_csv(path, nrows=0).columns)
    except Exception:
        return False
    return REQUIRED_COLUMNS.issubset(cols)


def collect_score_paths():
    rows = []
    for model, model_root in MODEL_DIRS.items():
        if not model_root.exists():
            continue
        for path in sorted(model_root.rglob('*.csv')):
            if has_required_columns(path):
                rows.append({
                    'model': model,
                    'path': path,
                    'relative_path': str(path.relative_to(ROOT)),
                })
    return pd.DataFrame(rows)


score_paths = collect_score_paths()
print('score csv files:', len(score_paths))
display(score_paths)


## 2. Load Rows and Extract Defect Type

In [ ]:
def normalize_path(path):
    return str(path).replace(chr(92), '/')


def defect_type_from_path(path):
    parts = normalize_path(path).split('/')
    if 'test' in parts:
        idx = parts.index('test')
        if idx + 1 < len(parts):
            return parts[idx + 1]
    if 'Images' in parts:
        idx = parts.index('Images')
        if idx + 1 < len(parts):
            defect_type = parts[idx + 1]
            return 'good' if defect_type.lower() == 'normal' else defect_type
    return 'unknown'


def load_score_file(row):
    df = pd.read_csv(row['path']).copy()
    df['model'] = row['model']
    df['source_file'] = row['relative_path']
    df['dataset'] = df['dataset'].astype(str).str.lower()
    df['class'] = df['class'].astype(str)
    df['label'] = df['label'].astype(int)
    df['image_score'] = df['image_score'].astype(float)
    if 'sample_id' in df.columns:
        df['sample_id'] = df['sample_id'].astype(str)
    if 'query_path' in df.columns:
        df['path_norm'] = df['query_path'].map(normalize_path)
        df['defect_type'] = df['query_path'].map(defect_type_from_path)
    else:
        df['path_norm'] = ''
        df['defect_type'] = np.where(df['label'].eq(0), 'good', 'unknown')
    return df


if score_paths.empty:
    raise ValueError('No score CSVs with required columns were found under results/adaptclip, results/anomalyclip, or results/winclip.')

scores = pd.concat([load_score_file(row) for _, row in score_paths.iterrows()], ignore_index=True)
scores.to_csv(OUT_DIR / 'all_sample_scores_raw.csv', index=False)

print('rows:', len(scores))
display(scores.groupby(['model', 'dataset', 'class', 'defect_type', 'label']).size().rename('n').reset_index())


## 3. Compute Defect-Type AUROC

Each row compares one defect type against the same class's good samples for the same model.

In [ ]:
rows = []
for (model, dataset, class_name), model_class_df in scores.groupby(['model', 'dataset', 'class'], sort=True):
    good = model_class_df[model_class_df['label'] == 0].copy()
    anomalies = model_class_df[model_class_df['label'] == 1].copy()
    if good.empty or anomalies.empty:
        continue

    class_auroc = roc_auc_score(model_class_df['label'], model_class_df['image_score']) * 100 if model_class_df['label'].nunique() == 2 else np.nan

    for defect_type, defect_df in anomalies.groupby('defect_type', sort=True):
        eval_df = pd.concat([good, defect_df], ignore_index=True)
        if eval_df['label'].nunique() < 2:
            defect_auroc = np.nan
        else:
            defect_auroc = roc_auc_score(eval_df['label'], eval_df['image_score']) * 100

        rows.append({
            'model': model,
            'dataset': dataset,
            'class': class_name,
            'defect_type': defect_type,
            'defect_auroc': defect_auroc,
            'class_auroc_reference': class_auroc,
            'n_good': len(good),
            'n_defect': len(defect_df),
            'mean_good_score': good['image_score'].mean(),
            'mean_defect_score': defect_df['image_score'].mean(),
            'score_gap': defect_df['image_score'].mean() - good['image_score'].mean(),
            'std_good_score': good['image_score'].std(ddof=1),
            'std_defect_score': defect_df['image_score'].std(ddof=1),
            'iqr_defect_score': defect_df['image_score'].quantile(0.75) - defect_df['image_score'].quantile(0.25),
            'range_defect_score': defect_df['image_score'].max() - defect_df['image_score'].min(),
            'top_normal_score': good['image_score'].max(),
            'bottom_anomaly_score': defect_df['image_score'].min(),
            'source_files': '; '.join(sorted(eval_df['source_file'].unique())),
        })

defect_summary = pd.DataFrame(rows)
defect_summary['weak_defect_type'] = defect_summary['defect_auroc'].lt(LOW_DEFECT_AUROC_THRESHOLD)
defect_summary['rank_low_to_high'] = (
    defect_summary.groupby(['model', 'dataset', 'class'])['defect_auroc']
    .rank(ascending=True, method='min')
    .astype('Int64')
)

sort_cols = ['dataset', 'class', 'model', 'defect_auroc', 'defect_type']
defect_summary['dataset'] = pd.Categorical(defect_summary['dataset'], DATASET_ORDER, ordered=True)
defect_summary['model'] = pd.Categorical(defect_summary['model'], MODEL_ORDER, ordered=True)
defect_summary = defect_summary.sort_values(sort_cols).reset_index(drop=True)
defect_summary['dataset'] = defect_summary['dataset'].astype(str)
defect_summary['model'] = defect_summary['model'].astype(str)

round_cols = [
    'defect_auroc', 'class_auroc_reference', 'mean_good_score', 'mean_defect_score', 'score_gap',
    'std_good_score', 'std_defect_score', 'iqr_defect_score', 'range_defect_score',
    'top_normal_score', 'bottom_anomaly_score',
]
defect_summary[round_cols] = defect_summary[round_cols].round(4)
defect_summary.to_csv(OUT_DIR / 'defect_type_auroc_summary.csv', index=False)
defect_summary


## 4. Pivot By Defect Type

In [ ]:
pivot = defect_summary.pivot_table(
    index=['dataset', 'class', 'defect_type'],
    columns='model',
    values='defect_auroc',
    aggfunc='first',
    observed=False,
)
model_cols = [model for model in MODEL_ORDER if model in pivot.columns]
pivot = pivot[model_cols].copy()
pivot['mean_defect_auroc'] = pivot[model_cols].mean(axis=1)
pivot['min_defect_auroc'] = pivot[model_cols].min(axis=1)
pivot['max_defect_auroc'] = pivot[model_cols].max(axis=1)
pivot['model_spread'] = pivot['max_defect_auroc'] - pivot['min_defect_auroc']
pivot['best_model'] = pivot[model_cols].idxmax(axis=1)
pivot['worst_model'] = pivot[model_cols].idxmin(axis=1)
pivot = pivot.round(4).reset_index()
pivot.to_csv(OUT_DIR / 'defect_type_auroc_pivot.csv', index=False)
pivot


## 5. Lowest Defect Types

In [ ]:
lowest_by_model = (
    defect_summary
    .sort_values(['model', 'defect_auroc', 'dataset', 'class', 'defect_type'])
    .groupby('model', observed=False)
    .head(30)
    .reset_index(drop=True)
)
lowest_by_model.to_csv(OUT_DIR / 'lowest_defect_types_by_model_top30.csv', index=False)
lowest_by_model


In [ ]:
common_low = pivot.sort_values(['mean_defect_auroc', 'min_defect_auroc']).reset_index(drop=True)
common_low.to_csv(OUT_DIR / 'lowest_mean_defect_types_across_models.csv', index=False)
common_low.head(50)


## 6. Dataset/Class Aggregates From Defect-Type AUROC

In [ ]:
class_defect_agg = (
    defect_summary
    .groupby(['model', 'dataset', 'class'], observed=False)
    .agg(
        n_defect_types=('defect_type', 'nunique'),
        mean_defect_auroc=('defect_auroc', 'mean'),
        median_defect_auroc=('defect_auroc', 'median'),
        min_defect_auroc=('defect_auroc', 'min'),
        max_defect_auroc=('defect_auroc', 'max'),
        n_weak_defect_types=('weak_defect_type', 'sum'),
    )
    .reset_index()
)
for col in ['mean_defect_auroc', 'median_defect_auroc', 'min_defect_auroc', 'max_defect_auroc']:
    class_defect_agg[col] = class_defect_agg[col].round(4)
class_defect_agg.to_csv(OUT_DIR / 'class_level_from_defect_type_auroc.csv', index=False)
class_defect_agg


## 7. Heatmaps

In [ ]:
heatmap_df = pivot.copy()
heatmap_df['target'] = heatmap_df['dataset'].astype(str) + '/' + heatmap_df['class'].astype(str) + '/' + heatmap_df['defect_type'].astype(str)
heatmap_values = heatmap_df.set_index('target')[model_cols]

fig, ax = plt.subplots(figsize=(max(6, 1.4 * len(model_cols)), max(5, 0.28 * len(heatmap_values))))
im = ax.imshow(heatmap_values.to_numpy(dtype=float), aspect='auto', vmin=0, vmax=100, cmap='viridis')
ax.set_xticks(np.arange(len(model_cols)))
ax.set_xticklabels(model_cols, rotation=30, ha='right')
ax.set_yticks(np.arange(len(heatmap_values)))
ax.set_yticklabels(heatmap_values.index, fontsize=8)
ax.set_title('Defect-type AUROC by model')

for i in range(heatmap_values.shape[0]):
    for j in range(heatmap_values.shape[1]):
        value = heatmap_values.iloc[i, j]
        if pd.notna(value):
            color = 'white' if value < 60 else 'black'
            ax.text(j, i, f'{value:.0f}', ha='center', va='center', fontsize=7, color=color)

fig.colorbar(im, ax=ax, label='Defect-type AUROC (%)')
plt.tight_layout()
plt.show()


In [ ]:
for dataset, dataset_df in defect_summary.groupby('dataset', sort=False):
    if dataset_df.empty:
        continue
    classes = sorted(dataset_df['class'].unique())
    for class_name in classes:
        class_df = dataset_df[dataset_df['class'] == class_name].copy()
        plot_df = class_df.pivot_table(index='defect_type', columns='model', values='defect_auroc', aggfunc='first', observed=False)
        plot_cols = [model for model in MODEL_ORDER if model in plot_df.columns]
        if not plot_cols:
            continue
        plot_df = plot_df[plot_cols].sort_index()
        ax = plot_df.plot(kind='bar', figsize=(max(7, 0.7 * len(plot_df)), 3.8), width=0.8)
        ax.axhline(LOW_DEFECT_AUROC_THRESHOLD, color='black', linestyle='--', linewidth=1)
        ax.set_title(f'{dataset}/{class_name}: defect-type AUROC')
        ax.set_ylabel('AUROC (%)')
        ax.set_ylim(0, 100)
        ax.tick_params(axis='x', rotation=45)
        for label in ax.get_xticklabels():
            label.set_ha('right')
        ax.legend(frameon=False, ncol=len(plot_cols))
        plt.tight_layout()
        plt.show()


## 8. Model Gap By Defect Type

In [ ]:
gap_view = pivot.sort_values('model_spread', ascending=False).reset_index(drop=True)
gap_view.to_csv(OUT_DIR / 'largest_model_defect_type_auroc_gaps.csv', index=False)
gap_view.head(50)


## 9. Output Files

Saved under `results/defect_type_overview/`:

- `all_sample_scores_raw.csv`
- `defect_type_auroc_summary.csv`
- `defect_type_auroc_pivot.csv`
- `lowest_defect_types_by_model_top30.csv`
- `lowest_mean_defect_types_across_models.csv`
- `class_level_from_defect_type_auroc.csv`
- `largest_model_defect_type_auroc_gaps.csv`
